[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/22_conv2d.ipynb)

# 🟠 Medium: 2D Convolution

Implement **2D convolution** from scratch.

### Signature
```python
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    # x: (B, C_in, H, W), weight: (C_out, C_in, kH, kW)
    # Returns: (B, C_out, H_out, W_out)
```

### Rules
- Do NOT use `F.conv2d` or `nn.Conv2d`
- Support `stride` and `padding` parameters
- `F.pad` for zero-padding is allowed

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn.functional as F

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [14]:
# ✏️ YOUR IMPLEMENTATION HERE
def my_conv2d(x, weight, bias=None, stride=1, padding=0):
    if padding > 0:
         x = F.pad(x, [padding] * 4)
    # extract patches, apply kernel, handle stride/padding
    batch_size, c_in, h_in, w_in = x.shape
    c_out, c_in, h_k, w_k = weight.shape
    x = x.unfold(2, h_k, stride).unfold(3, w_k, stride) # (batch_size, c_in, h_out, w_out, h_k, w_k)
    out = torch.einsum('bchwij, ocij -> bohw', x, weight)
    if bias is not None:
         out += bias.view(1, -1, 1, 1)
    return out

In [15]:
# 🧪 Debug
x = torch.randn(1, 3, 8, 8)
w = torch.randn(16, 3, 3, 3)
print('Output:', my_conv2d(x, w).shape)
print('Match:', torch.allclose(my_conv2d(x, w), F.conv2d(x, w), atol=1e-4))

Output: torch.Size([1, 16, 6, 6])
Match: True


In [16]:
# ✅ SUBMIT
from torch_judge import check
check('conv2d')


🧪 Testing: 2D Convolution (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (0.8ms)
  ✅ [2/5] Matches F.conv2d (3.1ms)
  ✅ [3/5] With padding (2.2ms)
  ✅ [4/5] With stride (1.2ms)
  ✅ [5/5] Gradient flow (0.6ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (7.8ms total)
  Progress saved. Run status() to see your dashboard.

